In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.stats import norm


In [2]:
def make_nf4_codebook():
    """
    Build the 16-value NF4 codebook: quantiles of a standard normal
    distribution, asymmetric (8 negative, exact 0, 7 positive) so that
    zero is exactly representable.
    """
    offset = 0.9677083
    neg_probs = torch.linspace(1 - offset, 0.5, 9)[:-1]
    neg = norm.ppf(neg_probs.tolist())
    pos_probs = torch.linspace(0.5, offset, 8)[1:]
    pos = norm.ppf(pos_probs.tolist())
    codebook = torch.tensor(list(neg) + [0.0] + list(pos), dtype=torch.float32)
    codebook = torch.sort(codebook).values
    codebook = codebook / codebook.abs().max()
    return codebook


In [3]:
def quantize_nf4(tensor, codebook, block_size=64):
    device = tensor.device
    codebook = codebook.to(device)
    flat = tensor.flatten()
    n = flat.numel()
    pad = (-n) % block_size
    if pad:
        flat = torch.cat([flat, torch.zeros(pad, device=device)])
    blocks = flat.view(-1, block_size)
    scales = blocks.abs().max(dim=1, keepdim=True).values.clamp(min=1e-8)
    normalized = blocks / scales
    diffs = (normalized.unsqueeze(-1) - codebook.view(1, 1, -1)).abs()
    indices = diffs.argmin(dim=-1).to(torch.uint8)
    return indices, scales.squeeze(1), tensor.shape, n

def dequantize_nf4(indices, scales, codebook, orig_shape, orig_numel, block_size=64):
    codebook = codebook.to(indices.device)
    looked_up = codebook[indices.long()]
    dequantized = looked_up * scales.unsqueeze(1)
    flat = dequantized.flatten()[:orig_numel]
    return flat.view(orig_shape)

def quantize_scales_8bit(scales, block_size=256):
    device = scales.device
    mean = scales.mean()
    centered = scales - mean
    n = centered.numel()
    pad = (-n) % block_size
    if pad:
        centered = torch.cat([centered, torch.zeros(pad, device=device)])
    blocks = centered.view(-1, block_size)
    block_scales = blocks.abs().max(dim=1, keepdim=True).values.clamp(min=1e-8)
    normalized = blocks / block_scales
    levels = torch.linspace(-1, 1, 256, device=device)
    diffs = (normalized.unsqueeze(-1) - levels.view(1, 1, -1)).abs()
    indices = diffs.argmin(dim=-1).to(torch.uint8)
    return indices, block_scales.squeeze(1), mean, n

def dequantize_scales_8bit(indices, block_scales, mean, orig_numel, block_size=256):
    levels = torch.linspace(-1, 1, 256, device=indices.device)
    looked_up = levels[indices.long()]
    dequantized = looked_up * block_scales.unsqueeze(1)
    flat = dequantized.flatten()[:orig_numel]
    return flat + mean

In [4]:
def naive_int4_quantize(tensor, block_size=64):
    flat = tensor.flatten()
    n = flat.numel()
    pad = (-n) % block_size
    if pad:
        flat = torch.cat([flat, torch.zeros(pad)])
    blocks = flat.view(-1, block_size)
    scales = blocks.abs().max(dim=1, keepdim=True).values.clamp(min=1e-8)
    normalized = blocks / scales

    levels = torch.linspace(-1, 1, 16)   # <-- the ONLY real difference from NF4
    diffs = (normalized.unsqueeze(-1) - levels.view(1, 1, -1)).abs()
    idx = diffs.argmin(dim=-1)

    dequant = levels[idx] * scales
    return dequant.flatten()[:n].view(tensor.shape)

In [5]:
if __name__ == "__main__":
    cb = make_nf4_codebook()
    print("Codebook:", cb)
 
    # --- small readable test (8x4, block_size=8) ---
    torch.manual_seed(0)
    test_tensor = torch.randn(4, 8) * 0.02
 
    indices, scales, shape, n = quantize_nf4(test_tensor, cb, block_size=8)
    reconstructed = dequantize_nf4(indices, scales, cb, shape, n, block_size=8)
 
    print("\n--- Small test tensor (4x8, block_size=8) ---")
    print("Original:\n", test_tensor)
    print("Indices:\n", indices)
    print("Scales:\n", scales)
    print("Reconstructed:\n", reconstructed)
    print("Max absolute error:", (test_tensor - reconstructed).abs().max().item())
 
    naive_reconstructed = naive_int4_quantize(test_tensor, block_size=8)
    nf4_mse_small = torch.mean((test_tensor - reconstructed) ** 2).item()
    int4_mse_small = torch.mean((test_tensor - naive_reconstructed) ** 2).item()
    print(f"\n[Small tensor, N=32 -- too few samples for the statistical NF4")
    print(f" advantage to show reliably, kept here only for illustration]")
    print(f"NF4 MSE: {nf4_mse_small:.10f}   Naive INT4 MSE: {int4_mse_small:.10f}")
 
    # --- large realistic test (1024x1024, block_size=64) ---
    torch.manual_seed(0)
    big_tensor = torch.randn(1024, 1024) * 0.02  # ~1M values, weight-matrix-like scale
 
    indices, scales, shape, n = quantize_nf4(big_tensor, cb, block_size=64)
    nf4_recon = dequantize_nf4(indices, scales, cb, shape, n, block_size=64)
    int4_recon = naive_int4_quantize(big_tensor, block_size=64)
 
    nf4_mse = torch.mean((big_tensor - nf4_recon) ** 2).item()
    int4_mse = torch.mean((big_tensor - int4_recon) ** 2).item()
 
    print("\n--- Large tensor (1024x1024 = 1,048,576 values, block_size=64) ---")
    print("Original (top-left 8x8 slice):\n", big_tensor[:8, :8])
    print("\nNF4 reconstructed (same slice):\n", nf4_recon[:8, :8])
    print("\nNaive INT4 reconstructed (same slice):\n", int4_recon[:8, :8])
 
    print(f"\n--- Full tensor stats (all {big_tensor.numel()} values) ---")
    print(f"NF4 MSE:        {nf4_mse:.10f}")
    print(f"Naive INT4 MSE: {int4_mse:.10f}")
    print(f"NF4 improvement: {(1 - nf4_mse/int4_mse)*100:.2f}%")
 
    # --- compression ratio check ---
    orig_bytes = big_tensor.numel() * 4  # fp32
    packed_bytes = (big_tensor.numel() * 4 // 8) + (scales.numel() * 4)  # 4-bit/val + fp32 scales
    print(f"\nOriginal (fp32): {orig_bytes/1e6:.2f} MB")
    print(f"NF4 packed (4-bit/val + scales): {packed_bytes/1e6:.2f} MB")
    print(f"Compression ratio: {orig_bytes/packed_bytes:.2f}x")
 

Codebook: tensor([-1.0000, -0.7230, -0.5626, -0.4407, -0.3379, -0.2461, -0.1609, -0.0796,
         0.0000,  0.0911,  0.1848,  0.2844,  0.3949,  0.5251,  0.6962,  1.0000])

--- Small test tensor (4x8, block_size=8) ---
Original:
 tensor([[-0.0225, -0.0230, -0.0050, -0.0087,  0.0170,  0.0138, -0.0063, -0.0423],
        [ 0.0064, -0.0253,  0.0070,  0.0062,  0.0024,  0.0248,  0.0223, -0.0049],
        [-0.0271, -0.0339,  0.0113,  0.0159,  0.0120, -0.0311, -0.0068,  0.0371],
        [ 0.0150, -0.0117, -0.0035,  0.0037,  0.0278,  0.0317,  0.0189, -0.0169]])
Indices:
 tensor([[ 2,  2,  7,  5, 12, 11,  6,  0],
        [11,  0, 11, 11,  9, 15, 15,  6],
        [ 1,  0, 11, 12, 11,  1,  6, 15],
        [13,  4,  7,  9, 15, 15, 13,  2]], dtype=torch.uint8)
Scales:
 tensor([0.0423, 0.0253, 0.0371, 0.0317])
Reconstructed:
 tensor([[-0.0238, -0.0238, -0.0034, -0.0104,  0.0167,  0.0120, -0.0068, -0.0423],
        [ 0.0072, -0.0253,  0.0072,  0.0072,  0.0023,  0.0253,  0.0253, -0.0041],
        [-0.02

In [6]:


class QuantizedLinear(nn.Module):
    def __init__(self, weight: torch.Tensor, bias: torch.Tensor, codebook: torch.Tensor,
                 block_size: int = 64, scale_block_size: int = 256):
        super().__init__()
        self.codebook = codebook
        self.block_size = block_size
        self.scale_block_size = scale_block_size
        self.out_features, self.in_features = weight.shape

        indices, scales, orig_shape, orig_numel = quantize_nf4(weight, codebook, block_size)
        scale_indices, scale_block_scales, scale_mean, scale_numel = quantize_scales_8bit(
            scales, scale_block_size
        )

        self.register_buffer("indices", indices)
        self.register_buffer("scale_indices", scale_indices)
        self.register_buffer("scale_block_scales", scale_block_scales)
        self.register_buffer("scale_mean", scale_mean)
        self.orig_shape = orig_shape
        self.orig_numel = orig_numel
        self.scale_numel = scale_numel

        if bias is not None:
            self.register_buffer("bias", bias)
        else:
            self.bias = None

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        scales = dequantize_scales_8bit(
            self.scale_indices, self.scale_block_scales, self.scale_mean,
            self.scale_numel, self.scale_block_size
        )
        weight = dequantize_nf4(
            self.indices, scales, self.codebook,
            self.orig_shape, self.orig_numel, self.block_size
        )
        return F.linear(x, weight.to(x.dtype), self.bias)


    @classmethod
    def from_linear(cls, linear: nn.Linear, codebook: torch.Tensor,
                     block_size: int = 64, scale_block_size: int = 256):
        return cls(linear.weight.data, linear.bias.data if linear.bias is not None else None,
                    codebook, block_size, scale_block_size)

In [7]:
import gc

def quantize_model_inplace(model, codebook, target_modules=["q_proj", "v_proj"]):
    num_layers = len(model.model.layers)
    
    for i, layer in enumerate(model.model.layers):
        attn = layer.self_attn
        
        for name in target_modules:
            orig = getattr(attn, name)
            setattr(attn, name, QuantizedLinear.from_linear(orig, codebook))
        
        gc.collect()
        torch.cuda.empty_cache()
        print(f"[{i+1}/{num_layers}] done")
    
    print("quantization complete.")


In [8]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_id = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

In [9]:
cb = make_nf4_codebook()
quantize_model_inplace(model, cb, target_modules=["q_proj", "v_proj"])


[1/28] done
[2/28] done
[3/28] done
[4/28] done
[5/28] done
[6/28] done
[7/28] done
[8/28] done
[9/28] done
[10/28] done
[11/28] done
[12/28] done
[13/28] done
[14/28] done
[15/28] done
[16/28] done
[17/28] done
[18/28] done
[19/28] done
[20/28] done
[21/28] done
[22/28] done
[23/28] done
[24/28] done
[25/28] done
[26/28] done
[27/28] done
[28/28] done
quantization complete.


In [10]:
import math

class LoRALinear(nn.Module):
    def __init__(self, quantized_linear: QuantizedLinear, r: int = 8, alpha: int = 16):
        super().__init__()

        self.base = quantized_linear
        self.r = r
        self.scaling = alpha / r

        in_features = quantized_linear.in_features
        out_features = quantized_linear.out_features

        self.lora_A = nn.Parameter(torch.empty(r, in_features))
        self.lora_B = nn.Parameter(torch.zeros(out_features, r))

        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))

        for param in self.base.parameters():
            param.requires_grad = False

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        base_out = self.base(x)
        lora_out = F.linear(
            F.linear(x, self.lora_A.to(device=x.device, dtype=x.dtype)),
            self.lora_B.to(device=x.device, dtype=x.dtype)
        ) * self.scaling
        return base_out + lora_out



In [11]:
def inject_lora(model, r=8, alpha=16, target_modules=["q_proj", "v_proj"]):
    num_layers = len(model.model.layers)

    for i, layer in enumerate(model.model.layers):
        attn = layer.self_attn

        for name in target_modules:
            orig = getattr(attn, name)
            setattr(attn, name, LoRALinear(orig, r=r, alpha=alpha))

        print(f"[{i+1}/{num_layers}] LoRA injected")

    print("LoRA injection complete.")


In [12]:
inject_lora(model, r=8, alpha=16, target_modules=["q_proj", "v_proj"])


[1/28] LoRA injected
[2/28] LoRA injected
[3/28] LoRA injected
[4/28] LoRA injected
[5/28] LoRA injected
[6/28] LoRA injected
[7/28] LoRA injected
[8/28] LoRA injected
[9/28] LoRA injected
[10/28] LoRA injected
[11/28] LoRA injected
[12/28] LoRA injected
[13/28] LoRA injected
[14/28] LoRA injected
[15/28] LoRA injected
[16/28] LoRA injected
[17/28] LoRA injected
[18/28] LoRA injected
[19/28] LoRA injected
[20/28] LoRA injected
[21/28] LoRA injected
[22/28] LoRA injected
[23/28] LoRA injected
[24/28] LoRA injected
[25/28] LoRA injected
[26/28] LoRA injected
[27/28] LoRA injected
[28/28] LoRA injected
LoRA injection complete.


In [13]:
# Step 1: Freeze every single parameter in the model
for param in model.parameters():
    param.requires_grad = False

# Step 2: Unfreeze only the LoRA adapter parameters
for name, param in model.named_parameters():
    if "lora_A" in name or "lora_B" in name:
        param.requires_grad = True

# Step 3: Verify
trainable = [(n, p.shape, p.numel()) for n, p in model.named_parameters() if p.requires_grad]
frozen    = [n for n, p in model.named_parameters() if not p.requires_grad]

total_trainable = sum(p[2] for p in trainable)
print(f"Trainable: {len(trainable)} params, {total_trainable:,} values")
print(f"Frozen:    {len(frozen)} params")
print()
for name, shape, count in trainable[:6]:
    print(f"  {name}  shape={shape}")


Trainable: 112 params, 2,523,136 values
Frozen:    227 params

  model.layers.0.self_attn.q_proj.lora_A  shape=torch.Size([8, 3584])
  model.layers.0.self_attn.q_proj.lora_B  shape=torch.Size([3584, 8])
  model.layers.0.self_attn.v_proj.lora_A  shape=torch.Size([8, 3584])
  model.layers.0.self_attn.v_proj.lora_B  shape=torch.Size([512, 8])
  model.layers.1.self_attn.q_proj.lora_A  shape=torch.Size([8, 3584])
  model.layers.1.self_attn.q_proj.lora_B  shape=torch.Size([3584, 8])


In [14]:
model.eval()

prompt = "SELECT * FROM"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model(**inputs)

print("Output logits shape:", outputs.logits.shape)
print("No NaN in logits:", not torch.isnan(outputs.logits).any().item())
print("Forward pass successful!")


Output logits shape: torch.Size([1, 3, 152064])
No NaN in logits: True
Forward pass successful!


In [15]:
trainable = []
frozen = []

for name, param in model.named_parameters():
    if param.requires_grad:
        trainable.append((name, param.shape, param.numel()))
    else:
        frozen.append(name)

print(f"Trainable parameters: {len(trainable)}")
print(f"Frozen parameters:    {len(frozen)}")
print()

total_trainable = sum(p[2] for p in trainable)
print(f"Total trainable param count: {total_trainable:,}")
print()

print("Trainable parameter names:")
for name, shape, count in trainable[:10]:
    print(f"  {name}  shape={shape}  count={count:,}")


Trainable parameters: 112
Frozen parameters:    227

Total trainable param count: 2,523,136

Trainable parameter names:
  model.layers.0.self_attn.q_proj.lora_A  shape=torch.Size([8, 3584])  count=28,672
  model.layers.0.self_attn.q_proj.lora_B  shape=torch.Size([3584, 8])  count=28,672
  model.layers.0.self_attn.v_proj.lora_A  shape=torch.Size([8, 3584])  count=28,672
  model.layers.0.self_attn.v_proj.lora_B  shape=torch.Size([512, 8])  count=4,096
  model.layers.1.self_attn.q_proj.lora_A  shape=torch.Size([8, 3584])  count=28,672
  model.layers.1.self_attn.q_proj.lora_B  shape=torch.Size([3584, 8])  count=28,672
  model.layers.1.self_attn.v_proj.lora_A  shape=torch.Size([8, 3584])  count=28,672
  model.layers.1.self_attn.v_proj.lora_B  shape=torch.Size([512, 8])  count=4,096
  model.layers.2.self_attn.q_proj.lora_A  shape=torch.Size([8, 3584])  count=28,672
  model.layers.2.self_attn.q_proj.lora_B  shape=torch.Size([3584, 8])  count=28,672


In [16]:
from datasets import load_dataset

# Load base Spider dataset
dataset = load_dataset("spider", trust_remote_code=True)
train_data = dataset["train"]
val_data = dataset["validation"]

# Load schema helper dataset
schema_dataset = load_dataset("richardr1126/spider-schema")
schema_lookup = {row["db_id"]: row for row in schema_dataset["train"]}

print(f"Loaded {len(train_data)} train and {len(val_data)} validation examples.")
print(f"Loaded schema lookup for {len(schema_lookup)} database schemas.")



`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'spider' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


README.md: 0.00B [00:00, ?B/s]

spider/train-00000-of-00001.parquet:   0%|          | 0.00/831k [00:00<?, ?B/s]

spider/validation-00000-of-00001.parquet:   0%|          | 0.00/126k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1034 [00:00<?, ? examples/s]

README.md: 0.00B [00:00, ?B/s]

spider_schema_rows_v2.json: 0.00B [00:00, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Loaded 7000 train and 1034 validation examples.
Loaded schema lookup for 166 database schemas.


In [17]:
def format_schema(db_id):
    row = schema_lookup[db_id]
    formatted = f"Schema:\n{row['Schema (values (type))']}"
    if row["Foreign Keys"]:
        formatted += f"\nForeign Keys:\n{row['Foreign Keys']}"
    return formatted

def build_prompt(question, schema_text):
    return f"""Given the database schema:
{schema_text}

Write a SQL query to answer: {question}

Respond with only the SQL query. No explanation, no markdown formatting."""

def tokenize_example(example, tokenizer, max_length=1024):
    """
    Constructs the prompt, tokenizes inputs and labels separately,
    and returns None if the example exceeds max_length to filter it out.
    """
    schema_text = format_schema(example["db_id"])
    prompt = build_prompt(example["question"], schema_text)
    
    messages = [{"role": "user", "content": prompt}]
    prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    answer_text = example["query"] + tokenizer.eos_token

    prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
    answer_ids = tokenizer(answer_text, add_special_tokens=False)["input_ids"]

    total_len = len(prompt_ids) + len(answer_ids)
    
    # Filter out examples exceeding max_length entirely
    if total_len > max_length:
        return None

    input_ids = prompt_ids + answer_ids
    labels = [-100] * len(prompt_ids) + answer_ids

    # Pad to max_length
    pad_len = max_length - len(input_ids)
    input_ids = input_ids + [tokenizer.pad_token_id] * pad_len
    labels    = labels    + [-100] * pad_len

    return {
        "input_ids": torch.tensor(input_ids),
        "labels":    torch.tensor(labels),
        "total_len": total_len,
    }


In [18]:
# 1. Tokenize train and val splits, filtering out None values (exceeding 1024 tokens)
train_raw_tokenized = [tokenize_example(ex, tokenizer, max_length=1024) for ex in train_data]
tokenized = [t for t in train_raw_tokenized if t is not None]

val_raw_tokenized = [tokenize_example(ex, tokenizer, max_length=1024) for ex in val_data]
val_tokenized = [t for t in val_raw_tokenized if t is not None]

# 2. Print counts
dropped_train = len(train_raw_tokenized) - len(tokenized)
dropped_val   = len(val_raw_tokenized) - len(val_tokenized)

print(f"Train split: {len(tokenized)} kept, {dropped_train} dropped.")
print(f"Val split:   {len(val_tokenized)} kept, {dropped_val} dropped.")

# 3. Inspect active lengths
lengths = [t["total_len"] for t in tokenized]
print(f"Max token length in kept: {max(lengths)}")
print(f"Average token length in kept: {sum(lengths)/len(lengths):.1f}")


Train split: 6836 kept, 164 dropped.
Val split:   1034 kept, 0 dropped.
Max token length in kept: 954
Average token length in kept: 359.3


In [19]:
from torch.utils.data import DataLoader

def collate_fn(batch):
    return {
        "input_ids": torch.stack([b["input_ids"] for b in batch]),
        "labels":    torch.stack([b["labels"]    for b in batch]),
    }

train_loader = DataLoader(
    tokenized,
    batch_size=2,
    shuffle=True,
    collate_fn=collate_fn
)

print(f"Total batches per epoch: {len(train_loader)}")


Total batches per epoch: 3418


In [20]:
val_data = dataset["validation"]
val_tokenized = [tokenize_example(ex, tokenizer) for ex in val_data]

val_loader = DataLoader(
    val_tokenized,
    batch_size=2,
    shuffle=False,
    collate_fn=collate_fn
)

print(f"Val examples: {len(val_tokenized)}")
print(f"Val batches:  {len(val_loader)}")


Val examples: 1034
Val batches:  517


In [ ]:
from transformers import get_linear_schedule_with_warmup
from torch.optim import AdamW
from tqdm.notebook import tqdm

# --- CRITICAL MEMORY FIXES FOR 1024 LENGTH ---
model.config.use_cache = False
model.gradient_checkpointing_enable()
torch.cuda.empty_cache()

# Optimizer (only trains LoRA weights)
optimizer = AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=1e-4
)

num_training_steps = 1000
num_warmup_steps   = int(0.1 * num_training_steps)  # 100 warmup steps
val_steps          = 50
log_every          = 25
accumulation_steps = 2  # Process 1 at a time, update weights every 2 steps

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps
)

def run_validation(model, val_loader, max_steps=50):
    model.eval()
    total_val_loss = 0.0
    with torch.no_grad():
        for i, batch in enumerate(val_loader):
            if i >= max_steps:
                break
            input_ids = batch["input_ids"].to(model.device)
            labels    = batch["labels"].to(model.device)
            total_val_loss += model(input_ids=input_ids, labels=labels).loss.item()
    model.train()
    return total_val_loss / max_steps

best_val_loss    = float("inf")
total_train_loss = 0.0

model.train()
optimizer.zero_grad()

for step, batch in enumerate(tqdm(train_loader, desc="Training", total=num_training_steps)):
    if step >= num_training_steps:
        break

    input_ids = batch["input_ids"].to(model.device)
    labels    = batch["labels"].to(model.device)

    # Forward pass
    outputs = model(input_ids=input_ids, labels=labels)
    loss = outputs.loss / accumulation_steps
    loss.backward()

    total_train_loss += outputs.loss.item()

    # Step the optimizer and scheduler only after accumulation steps
    if (step + 1) % accumulation_steps == 0:
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

    # Evaluation and checkpoint saving
    if (step + 1) % log_every == 0:
        avg_train = total_train_loss / (step + 1)
        val_loss  = run_validation(model, val_loader, max_steps=val_steps)
        print(
            f"\nStep {step+1}/{num_training_steps} | "
            f"train: {avg_train:.4f} | val: {val_loss:.4f}",
            flush=True
        )
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save({
                "lora_state_dict": {k: v for k, v in model.state_dict().items() if "lora" in k},
                "step": step + 1,
                "val_loss": val_loss,
            }, "/kaggle/working/best_qlora_checkpoint.pt")
            print(f"  -> new best saved (val: {val_loss:.4f})", flush=True)

print("\nTraining complete.", flush=True)
print(f"Best val loss: {best_val_loss:.4f} (see /kaggle/working/best_qlora_checkpoint.pt)")


Training:   0%|          | 0/1000 [00:00<?, ?it/s]


Step 25/1000 | train: 1.4278 | val: 1.4214
  -> new best saved (val: 1.4214)

Step 50/1000 | train: 1.3603 | val: 1.2464
  -> new best saved (val: 1.2464)

Step 75/1000 | train: 1.2315 | val: 0.9983
  -> new best saved (val: 0.9983)

Step 100/1000 | train: 1.0747 | val: 0.5929
  -> new best saved (val: 0.5929)

Step 125/1000 | train: 0.9151 | val: 0.3651
  -> new best saved (val: 0.3651)

Step 150/1000 | train: 0.8175 | val: 0.3195
  -> new best saved (val: 0.3195)

Step 175/1000 | train: 0.7508 | val: 0.2965
  -> new best saved (val: 0.2965)

Step 200/1000 | train: 0.6972 | val: 0.2803
  -> new best saved (val: 0.2803)

Step 225/1000 | train: 0.6466 | val: 0.2726
  -> new best saved (val: 0.2726)

Step 250/1000 | train: 0.6062 | val: 0.2717
  -> new best saved (val: 0.2717)

Step 275/1000 | train: 0.5694 | val: 0.2728

Step 300/1000 | train: 0.5425 | val: 0.2580
  -> new best saved (val: 0.2580)

Step 325/1000 | train: 0.5182 | val: 0.2461
  -> new best saved (val: 0.2461)

Step 350/

In [22]:
checkpoint = torch.load("/kaggle/working/best_qlora_checkpoint.pt", map_location=model.device)
lora_state = checkpoint["lora_state_dict"]

for name, param in model.named_parameters():
    if name in lora_state:
        param.data.copy_(lora_state[name].to(param.device))

print(f"Loaded checkpoint from step {checkpoint['step']} (val_loss: {checkpoint['val_loss']:.4f})")


Loaded checkpoint from step 350 (val_loss: 0.2434)


In [27]:
import random
import re
from collections import Counter


def generate_sql(question, db_id, model, tokenizer, max_new_tokens=50):
    schema_text = format_schema(db_id)
    prompt = build_prompt(question, schema_text)
    
    messages = [{"role": "user", "content": prompt}]
    prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

# --- NEW NORMALIZATION LAYER ---
def normalize_sql(sql: str) -> str:
    """
    Normalize a SQL string so that semantically-identical queries with only
    surface-level formatting differences compare equal.
    """
    s = sql.strip()

    # Normalize whitespace: collapse multiple spaces/newlines to single space
    s = re.sub(r"\s+", " ", s)

    # Normalize quote style: convert double quotes to single quotes
    s = re.sub(r'"([^"]*)"', r"'\1'", s)

    # Lowercase everything (case-insensitive identifiers)
    s = s.lower()

    # Normalize spacing around commas and parentheses
    s = re.sub(r"\s*,\s*", ", ", s)
    s = re.sub(r"\s*\(\s*", "(", s)
    s = re.sub(r"\s*\)\s*", ")", s)

    # Remove trailing semicolons
    s = s.rstrip(";").strip()

    return s

def exact_match_normalized(gold: str, pred: str) -> bool:
    return normalize_sql(gold) == normalize_sql(pred)


def analyze_error(pred, gold):
    """
    Compares the predicted and gold SQL strings to pinpoint structural discrepancies.
    """
    pred_lower = pred.lower().replace(" ,", ",").replace(", ", ",").replace(" (", "(").replace("( ", "(")
    gold_lower = gold.lower().replace(" ,", ",").replace(", ", ",").replace(" (", "(").replace("( ", "(")
    
    mismatches = []
    
    # 1. Check structural clauses
    clauses = ["join", "where", "group by", "order by", "limit", "intersect", "union", "except"]
    for clause in clauses:
        pred_has = clause in pred_lower
        gold_has = clause in gold_lower
        if pred_has != gold_has:
            if gold_has:
                mismatches.append(f"Missing {clause.upper()} clause")
            else:
                mismatches.append(f"Redundant {clause.upper()} clause")
                
    # 2. Check aggregations
    aggs = ["count(", "avg(", "sum(", "min(", "max("]
    for agg in aggs:
        pred_has = agg in pred_lower
        gold_has = agg in gold_lower
        if pred_has != gold_has:
            if gold_has:
                mismatches.append(f"Missing {agg[:-1].upper()} aggregation")
            else:
                mismatches.append(f"Redundant {agg[:-1].upper()} aggregation")

    # 3. Check for specific identifier mismatches (columns or table names)
    if not mismatches:
        pred_words = set(pred_lower.replace(".", " ").replace("(", " ").replace(")", " ").split())
        gold_words = set(gold_lower.replace(".", " ").replace("(", " ").replace(")", " ").split())
        missing_ids = gold_words - pred_words
        extra_ids = pred_words - gold_words
        
        # Filter out common SQL syntax tokens to isolate columns/tables
        sql_keywords = {
            "select", "from", "where", "join", "on", "and", "or", "as", "group", "by", 
            "order", "limit", "in", "not", "t1", "t2", "t3", "t4", "t5", "=", ">", "<", 
            "count", "avg", "sum", "min", "max", "desc", "asc", "distinct", "having"
        }
        missing_ids = [w for w in missing_ids if w not in sql_keywords and not w.isdigit()]
        extra_ids = [w for w in extra_ids if w not in sql_keywords and not w.isdigit()]
        
        if missing_ids:
            mismatches.append(f"Missing/Wrong Columns or Tables: {missing_ids}")
        elif extra_ids:
            mismatches.append(f"Hallucinated Columns or Tables: {extra_ids}")
            
    if not mismatches:
        mismatches.append("Minor syntax / alias / spacing differences")
        
    return mismatches

# Run Evaluation
model.eval()
random.seed(42)
eval_subset = random.sample(list(dataset["validation"]), 300)

strict_matches     = 0
normalized_matches = 0
results            = []

# Iterate quietly with tqdm
for example in tqdm(eval_subset, desc="Evaluating 300 examples"):
    gold = example["query"]
    pred = generate_sql(example["question"], example["db_id"], model, tokenizer)
    
    # Run both checks
    strict_match = gold.strip() == pred.strip()
    normalized_match = exact_match_normalized(gold, pred)
    
    if strict_match:
        strict_matches += 1
    if normalized_match:
        normalized_matches += 1
    
    # Run error analysis if there's no normalized match
    error_reasons = []
    if not normalized_match:
        error_reasons = analyze_error(pred, gold)
        
    results.append({
        "question": example["question"],
        "gold":     gold,
        "pred":     pred,
        "match":    normalized_match,
        "errors":   error_reasons
    })

strict_accuracy = strict_matches / 300 * 100
normalized_accuracy = normalized_matches / 300 * 100

print(f"\n======================================")
print(f"EVALUATION SUMMARY")
print(f"======================================")
print(f"Strict Exact Match     : {strict_matches} / 300 = {strict_accuracy:.2f}%")
print(f"Normalized Exact Match : {normalized_matches} / 300 = {normalized_accuracy:.2f}%")
print(f"Total Errors (Normalized): {300 - normalized_matches}")

# Aggregate error categories across all failed predictions
all_errors = []
for r in results:
    if not r["match"]:
        for err in r["errors"]:
            if "Columns or Tables" in err:
                all_errors.append("Column/Table Name Mismatch")
            else:
                all_errors.append(err)

error_counts = Counter(all_errors)
print(f"\n======================================")
print(f"ERROR CATEGORY BREAKDOWN")
print(f"======================================")
for err_type, count in error_counts.most_common():
    print(f"  {err_type:<40} : {count} occurrences")

# Show top 5 detailed examples of failures
print(f"\n======================================")
print(f"SAMPLE ERROR DETAILS (First 5 Failures)")
print(f"======================================")
fail_idx = 1
for r in results:
    if not r["match"]:
        print(f"Failure #{fail_idx}")
        print(f"  Q    : {r['question']}")
        print(f"  Gold : {r['gold']}")
        print(f"  Pred : {r['pred']}")
        print(f"  Issue: {', '.join(r['errors'])}")
        print("-" * 50)
        fail_idx += 1
        if fail_idx > 5:
            break


Evaluating 300 examples:   0%|          | 0/300 [00:00<?, ?it/s]


EVALUATION SUMMARY
Strict Exact Match     : 32 / 300 = 10.67%
Normalized Exact Match : 132 / 300 = 44.00%
Total Errors (Normalized): 168

ERROR CATEGORY BREAKDOWN
  Column/Table Name Mismatch               : 56 occurrences
  Redundant JOIN clause                    : 32 occurrences
  Minor syntax / alias / spacing differences : 28 occurrences
  Missing JOIN clause                      : 22 occurrences
  Missing LIMIT clause                     : 15 occurrences
  Redundant WHERE clause                   : 12 occurrences
  Missing ORDER BY clause                  : 11 occurrences
  Missing COUNT aggregation                : 9 occurrences
  Missing EXCEPT clause                    : 7 occurrences
  Redundant MAX aggregation                : 6 occurrences
  Missing GROUP BY clause                  : 5 occurrences
  Redundant GROUP BY clause                : 5 occurrences
  Redundant MIN aggregation                : 4 occurrences
  Missing WHERE clause                     : 4 occurrences
 